In [ ]:
!pip install -q transformers accelerate lightgbm

In [ ]:
# ============================================================
# PHASE 1.2E — PROTBERT SLIDING-WINDOW EMBEDDING BASELINE
# Colab CUDA/T4 optimized
# Model: Rostlab/prot_bert
# ============================================================

import os
import re
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)

import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", None)

print("--- Hardware Check ---")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Initial GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("Initial GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

--- Hardware Check ---
Torch version: 2.11.0+cu128
CUDA available: True
Using device: cuda
GPU: NVIDIA L4
Initial GPU memory allocated: 0.0 MB
Initial GPU memory reserved: 0.0 MB


In [ ]:
# ============================================================
# GOOGLE DRIVE PATHS
# ============================================================
import os
from pathlib import Path
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

# 2. Định nghĩa lại đường dẫn gốc
PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

# Tự động phát hiện nếu bạn viết hoa chữ 'Model' hoặc viết thường chữ 'model'
actual_model_folder = "model"
if not (PROJECT_DIR / actual_model_folder).exists() and (PROJECT_DIR / "Model").exists():
    actual_model_folder = "Model"
SPLIT_DIR = PROJECT_DIR / "model" / "phase1_protein_baseline" / "splits"
BASE_DIR = PROJECT_DIR / "model" / "phase1_2_protbert_sliding_window_embedding"

EMBED_DIR = BASE_DIR / "embeddings"
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

for folder in [BASE_DIR, EMBED_DIR, RESULT_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Split dir:", SPLIT_DIR)
print("Output dir:", BASE_DIR)

# 5. Kiểm tra an toàn để chắc chắn không bị lỗi AssertionError nữa
if not SPLIT_DIR.exists():
    print("\n❌ THÔNG BÁO: Vẫn không thấy thư mục dữ liệu splits.")
    print("Danh sách các mục thực tế đang có trong thư mục gốc dự án của bạn:")
    if PROJECT_DIR.exists():
        print(os.listdir(PROJECT_DIR))
    else:
        print(f"Không tìm thấy cả thư mục gốc: {PROJECT_DIR}")
else:
    print("\n✅ THÀNH CÔNG: Đường dẫn hoàn toàn hợp lệ! Hãy chạy tiếp các ô lệnh sau.")

Mounted at /content/drive
Split dir: /content/drive/MyDrive/Project_Protein/model/phase1_protein_baseline/splits
Output dir: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding

✅ THÀNH CÔNG: Đường dẫn hoàn toàn hợp lệ! Hãy chạy tiếp các ô lệnh sau.


In [ ]:
# ============================================================
# LOAD SAVED SPLITS
# ============================================================

train_path = SPLIT_DIR / "train_protein_v1.csv"
val_path = SPLIT_DIR / "val_protein_v1.csv"
test_path = SPLIT_DIR / "test_protein_v1.csv"

assert train_path.exists(), train_path
assert val_path.exists(), val_path
assert test_path.exists(), test_path

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Train: (1274, 35)
Validation: (273, 35)
Test: (273, 35)

Train labels:
label
0    637
1    637
Name: count, dtype: int64

Validation labels:
label
1    137
0    136
Name: count, dtype: int64

Test labels:
label
0    137
1    136
Name: count, dtype: int64


In [ ]:
# ============================================================
# CLEAN SEQUENCES
# Keep only 20 standard amino acids for consistency
# ============================================================

STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = "".join([aa for aa in seq if aa in STANDARD_AAS])
    return seq


def prepare_protbert_sequence(seq):
    """
    ProtBERT expects space-separated amino acids.
    Example:
        "MKT" -> "M K T"
    """
    seq = str(seq)
    return " ".join(list(seq))


for split_df in [train_df, val_df, test_df]:
    split_df["sequence_clean"] = split_df["sequence"].apply(clean_sequence)
    split_df["sequence_clean_length"] = split_df["sequence_clean"].str.len()

print("Train length summary:")
display(train_df["sequence_clean_length"].describe())

print("Validation length summary:")
display(val_df["sequence_clean_length"].describe())

print("Test length summary:")
display(test_df["sequence_clean_length"].describe())

Train length summary:


,sequence_clean_length
count,1274.000000
mean,744.512559
std,643.266085
min,41.000000
25%,354.000000
50%,555.000000
75%,926.000000
max,7388.000000


Validation length summary:


,sequence_clean_length
count,273.000000
mean,869.728938
std,2114.423720
min,56.000000
25%,370.000000
50%,576.000000
75%,977.000000
max,34350.000000


Test length summary:


,sequence_clean_length
count,273.000000
mean,774.432234
std,689.578849
min,51.000000
25%,326.000000
50%,574.000000
75%,1035.000000
max,4861.000000


In [ ]:
# ============================================================
# LOAD ProtBERT MODEL
# ============================================================

PROTBERT_MODEL_NAME = "Rostlab/prot_bert"
MODEL_SHORT_NAME = "ProtBERT"
REPRESENTATION_NAME = "ProtBERT_sliding_window_mean_pool_1022_stride_1022"

print("Loading tokenizer:", PROTBERT_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(
    PROTBERT_MODEL_NAME,
    do_lower_case=False
)

print("Loading model:", PROTBERT_MODEL_NAME)

if device.type == "cuda":
    protbert_model = AutoModel.from_pretrained(
        PROTBERT_MODEL_NAME,
        torch_dtype=torch.float16
    )
else:
    protbert_model = AutoModel.from_pretrained(PROTBERT_MODEL_NAME)

protbert_model.to(device)
protbert_model.eval()

print("Model loaded.")
print("Representation:", REPRESENTATION_NAME)
print("Device:", device)

if device.type == "cuda":
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

Loading tokenizer: Rostlab/prot_bert


config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Model loaded.
Representation: ProtBERT_sliding_window_mean_pool_1022_stride_1022
Device: cuda
GPU memory allocated: 801.5654296875 MB
GPU memory reserved: 822.0 MB


In [ ]:
# ============================================================
# SLIDING-WINDOW CONFIG
# ============================================================

WINDOW_SIZE = 1022
STRIDE = 1022

WINDOW_BATCH_SIZE = 1     # ProtBERT is heavy. If stable, try 2.
SAVE_EVERY = 25           # checkpoint every 25 proteins

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)
print("Window batch size:", WINDOW_BATCH_SIZE)
print("Save every:", SAVE_EVERY)
print("Representation:", REPRESENTATION_NAME)

Window size: 1022
Stride: 1022
Window batch size: 1
Save every: 25
Representation: ProtBERT_sliding_window_mean_pool_1022_stride_1022


In [ ]:
# ============================================================
# WINDOWING FUNCTION
# ============================================================

def split_sequence_into_windows(seq, window_size=1022, stride=1022):
    """
    Split a full protein sequence into non-overlapping windows.

    Example:
        length 2500, window_size 1022, stride 1022
        -> [1022, 1022, 456]
    """
    seq = str(seq)

    if len(seq) == 0:
        raise ValueError("Empty sequence after cleaning.")

    if len(seq) <= window_size:
        return [seq]

    windows = []
    start = 0

    while start < len(seq):
        end = start + window_size
        window = seq[start:end]

        if len(window) > 0:
            windows.append(window)

        start += stride

    return windows


# Quick test
example_seq = "A" * 2500
example_windows = split_sequence_into_windows(example_seq, WINDOW_SIZE, STRIDE)

print("Example number of windows:", len(example_windows))
print("Window lengths:", [len(w) for w in example_windows])

Example number of windows: 3
Window lengths: [1022, 1022, 456]


In [ ]:
# ============================================================
# BATCH WINDOW EMBEDDING FOR PROTBERT
# ============================================================

@torch.no_grad()
def embed_protbert_window_batch(
    window_seqs,
    tokenizer,
    model,
    device
):
    """
    Embed a batch of ProtBERT windows.

    Input:
        window_seqs: list[str], each raw amino-acid string <= 1022 aa

    Output:
        np.ndarray [n_windows, hidden_dim]
    """
    prepared_windows = [
        prepare_protbert_sequence(seq)
        for seq in window_seqs
    ]

    encoded = tokenizer(
        prepared_windows,
        return_tensors="pt",
        add_special_tokens=True,
        padding=True,
        truncation=True
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    if device.type == "cuda":
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(**encoded)
    else:
        outputs = model(**encoded)

    hidden = outputs.last_hidden_state
    attention_mask = encoded["attention_mask"]

    batch_embeddings = []

    for i in range(hidden.shape[0]):
        mask = attention_mask[i].bool()
        valid_positions = torch.where(mask)[0]

        # Exclude CLS and SEP if possible
        if len(valid_positions) > 2:
            aa_positions = valid_positions[1:-1]
        else:
            aa_positions = valid_positions

        aa_hidden = hidden[i, aa_positions, :]
        emb = aa_hidden.mean(dim=0)

        batch_embeddings.append(emb.detach().float().cpu().numpy())

    batch_embeddings = np.vstack(batch_embeddings)

    del encoded, outputs, hidden, attention_mask

    return batch_embeddings

In [ ]:
# ============================================================
# EMBED FULL PROTEIN USING SLIDING WINDOWS
# ============================================================

@torch.no_grad()
def embed_one_protein_protbert_sliding_window(
    seq,
    tokenizer,
    model,
    device,
    window_size=1022,
    stride=1022,
    window_batch_size=1
):
    """
    Full protein embedding:
    1. Split sequence into windows
    2. Embed each window
    3. Average window embeddings
    """
    windows = split_sequence_into_windows(
        seq,
        window_size=window_size,
        stride=stride
    )

    all_window_embeddings = []

    for start in range(0, len(windows), window_batch_size):
        batch_windows = windows[start:start + window_batch_size]

        batch_emb = embed_protbert_window_batch(
            window_seqs=batch_windows,
            tokenizer=tokenizer,
            model=model,
            device=device
        )

        all_window_embeddings.append(batch_emb)

    all_window_embeddings = np.vstack(all_window_embeddings)

    protein_embedding = all_window_embeddings.mean(axis=0)

    return protein_embedding, len(windows)

In [ ]:
# ============================================================
# DEBUG ONE SEQUENCE
# ============================================================

debug_seq = train_df.iloc[0]["sequence_clean"]

debug_emb, debug_n_windows = embed_one_protein_protbert_sliding_window(
    seq=debug_seq,
    tokenizer=tokenizer,
    model=protbert_model,
    device=device,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE
)

print("Debug embedding shape:", debug_emb.shape)
print("Number of windows:", debug_n_windows)

if device.type == "cuda":
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

Debug embedding shape: (1024,)
Number of windows: 1
GPU memory allocated: 810.6904296875 MB
GPU memory reserved: 828.0 MB


In [ ]:
# ============================================================
# ROBUST SPLIT EXTRACTION WITH RESUME
# ============================================================

def extract_protbert_sliding_embeddings_for_split_gpu(
    split_df,
    split_name,
    tokenizer,
    model,
    device,
    output_dir,
    sequence_col="sequence_clean",
    window_size=1022,
    stride=1022,
    window_batch_size=1,
    save_every=25
):
    output_dir = Path(output_dir)

    final_embedding_path = output_dir / f"protbert_sw_{split_name}_embeddings.npy"
    final_metadata_path = output_dir / f"protbert_sw_{split_name}_metadata.csv"

    partial_embedding_path = output_dir / f"protbert_sw_{split_name}_embeddings_partial.npy"
    partial_metadata_path = output_dir / f"protbert_sw_{split_name}_metadata_partial.csv"

    # If final files already exist, load them.
    if final_embedding_path.exists() and final_metadata_path.exists():
        print(f"[{split_name}] Final embeddings already exist. Loading...")
        embeddings = np.load(final_embedding_path)
        metadata = pd.read_csv(final_metadata_path)
        return embeddings, metadata

    split_df_reset = split_df.reset_index(drop=True)

    embeddings = []
    metadata_records = []
    start_idx = 0

    # Resume from partial checkpoint
    if partial_embedding_path.exists() and partial_metadata_path.exists():
        print(f"[{split_name}] Partial checkpoint found. Resuming...")

        partial_embeddings = np.load(partial_embedding_path)
        partial_metadata = pd.read_csv(partial_metadata_path)

        embeddings = [partial_embeddings[i] for i in range(partial_embeddings.shape[0])]
        metadata_records = partial_metadata.to_dict("records")

        start_idx = len(metadata_records)

        print(f"[{split_name}] Resuming from index {start_idx}")

    total_windows_processed = sum(
        record.get("n_windows", 0) for record in metadata_records
    )

    start_time = time.time()

    for idx in range(start_idx, len(split_df_reset)):
        row = split_df_reset.iloc[idx]
        seq = row[sequence_col]

        try:
            emb, n_windows = embed_one_protein_protbert_sliding_window(
                seq=seq,
                tokenizer=tokenizer,
                model=model,
                device=device,
                window_size=window_size,
                stride=stride,
                window_batch_size=window_batch_size
            )

            embeddings.append(emb)
            total_windows_processed += n_windows

            metadata_records.append({
                "row_index": idx,
                "gene_id": row.get("gene_id", None),
                "gene_symbol": row.get("gene_symbol", None),
                "uniprot_accession": row.get("uniprot_accession", None),
                "label": int(row["label"]),
                "sequence_clean_length": len(seq),
                "n_windows": n_windows,
                "window_size": window_size,
                "stride": stride,
                "representation": REPRESENTATION_NAME
            })

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"CUDA OOM at {split_name} index {idx}.")
                print("Try reducing WINDOW_BATCH_SIZE to 1.")
                if device.type == "cuda":
                    torch.cuda.empty_cache()
            raise e

        except Exception as e:
            print(f"Error at {split_name} index {idx}: {e}")
            raise e

        processed = len(metadata_records)

        # Checkpoint
        if processed % save_every == 0 or processed == len(split_df_reset):
            current_embeddings = np.vstack(embeddings)
            current_metadata = pd.DataFrame(metadata_records)

            np.save(partial_embedding_path, current_embeddings)
            current_metadata.to_csv(partial_metadata_path, index=False)

            elapsed = time.time() - start_time
            avg_windows = total_windows_processed / processed

            print(
                f"[{split_name}] {processed}/{len(split_df_reset)} proteins "
                f"| embeddings={current_embeddings.shape} "
                f"| total_windows={total_windows_processed} "
                f"| avg_windows/protein={avg_windows:.2f} "
                f"| elapsed={elapsed/60:.2f} min"
            )

            if device.type == "cuda":
                torch.cuda.empty_cache()

    final_embeddings = np.vstack(embeddings)
    final_metadata = pd.DataFrame(metadata_records)

    np.save(final_embedding_path, final_embeddings)
    final_metadata.to_csv(final_metadata_path, index=False)

    print(f"[{split_name}] Final embeddings shape:", final_embeddings.shape)
    print(f"[{split_name}] Saved:", final_embedding_path)
    print(f"[{split_name}] Metadata saved:", final_metadata_path)

    return final_embeddings, final_metadata

In [ ]:
# ============================================================
# DEBUG RUN — 10 TRAIN PROTEINS
# ============================================================

debug_df = train_df.head(10).copy()

X_debug_sw_protbert, meta_debug_sw_protbert = extract_protbert_sliding_embeddings_for_split_gpu(
    split_df=debug_df,
    split_name="debug10",
    tokenizer=tokenizer,
    model=protbert_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE,
    save_every=5
)

print("Debug embedding shape:", X_debug_sw_protbert.shape)
display(meta_debug_sw_protbert)

if device.type == "cuda":
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

[debug10] 5/10 proteins | embeddings=(5, 1024) | total_windows=5 | avg_windows/protein=1.00 | elapsed=0.00 min
[debug10] 10/10 proteins | embeddings=(10, 1024) | total_windows=10 | avg_windows/protein=1.00 | elapsed=0.01 min
[debug10] Final embeddings shape: (10, 1024)
[debug10] Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/embeddings/protbert_sw_debug10_embeddings.npy
[debug10] Metadata saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/embeddings/protbert_sw_debug10_metadata.csv
Debug embedding shape: (10, 1024)


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,n_windows,window_size,stride,representation
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
2,2,ENSG00000143167,GPA33,Q99795,0,319,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
5,5,ENSG00000213996,TM6SF2,Q9BZW4,1,377,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
6,6,ENSG00000164823,OSGIN2,Q9Y236,0,505,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
7,7,ENSG00000185716,MOSMO,Q8NHV5,0,167,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
8,8,ENSG00000181873,IBA57,Q5T440,0,356,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
9,9,ENSG00000164647,STEAP1,Q9UHE8,1,339,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022


GPU memory allocated: 810.6904296875 MB
GPU memory reserved: 824.0 MB


In [ ]:
# ============================================================
# EXTRACT VALIDATION SLIDING-WINDOW EMBEDDINGS
# ============================================================

X_val_sw_protbert, meta_val_sw_protbert = extract_protbert_sliding_embeddings_for_split_gpu(
    split_df=val_df,
    split_name="val",
    tokenizer=tokenizer,
    model=protbert_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE,
    save_every=SAVE_EVERY
)

[val] 25/273 proteins | embeddings=(25, 1024) | total_windows=32 | avg_windows/protein=1.28 | elapsed=0.02 min
[val] 50/273 proteins | embeddings=(50, 1024) | total_windows=70 | avg_windows/protein=1.40 | elapsed=0.03 min
[val] 75/273 proteins | embeddings=(75, 1024) | total_windows=103 | avg_windows/protein=1.37 | elapsed=0.05 min
[val] 100/273 proteins | embeddings=(100, 1024) | total_windows=138 | avg_windows/protein=1.38 | elapsed=0.06 min
[val] 125/273 proteins | embeddings=(125, 1024) | total_windows=172 | avg_windows/protein=1.38 | elapsed=0.08 min
[val] 150/273 proteins | embeddings=(150, 1024) | total_windows=206 | avg_windows/protein=1.37 | elapsed=0.10 min
[val] 175/273 proteins | embeddings=(175, 1024) | total_windows=269 | avg_windows/protein=1.54 | elapsed=0.13 min
[val] 200/273 proteins | embeddings=(200, 1024) | total_windows=297 | avg_windows/protein=1.49 | elapsed=0.14 min
[val] 225/273 proteins | embeddings=(225, 1024) | total_windows=328 | avg_windows/protein=1.46 |

In [ ]:
# ============================================================
# EXTRACT TEST SLIDING-WINDOW EMBEDDINGS
# ============================================================

X_test_sw_protbert, meta_test_sw_protbert = extract_protbert_sliding_embeddings_for_split_gpu(
    split_df=test_df,
    split_name="test",
    tokenizer=tokenizer,
    model=protbert_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE,
    save_every=SAVE_EVERY
)

[test] 25/273 proteins | embeddings=(25, 1024) | total_windows=30 | avg_windows/protein=1.20 | elapsed=0.01 min
[test] 50/273 proteins | embeddings=(50, 1024) | total_windows=61 | avg_windows/protein=1.22 | elapsed=0.03 min
[test] 75/273 proteins | embeddings=(75, 1024) | total_windows=97 | avg_windows/protein=1.29 | elapsed=0.05 min
[test] 100/273 proteins | embeddings=(100, 1024) | total_windows=124 | avg_windows/protein=1.24 | elapsed=0.06 min
[test] 125/273 proteins | embeddings=(125, 1024) | total_windows=158 | avg_windows/protein=1.26 | elapsed=0.07 min
[test] 150/273 proteins | embeddings=(150, 1024) | total_windows=189 | avg_windows/protein=1.26 | elapsed=0.09 min
[test] 175/273 proteins | embeddings=(175, 1024) | total_windows=224 | avg_windows/protein=1.28 | elapsed=0.10 min
[test] 200/273 proteins | embeddings=(200, 1024) | total_windows=263 | avg_windows/protein=1.31 | elapsed=0.12 min
[test] 225/273 proteins | embeddings=(225, 1024) | total_windows=297 | avg_windows/protei

In [ ]:
# ============================================================
# EXTRACT TRAIN SLIDING-WINDOW EMBEDDINGS
# ============================================================

X_train_sw_protbert, meta_train_sw_protbert = extract_protbert_sliding_embeddings_for_split_gpu(
    split_df=train_df,
    split_name="train",
    tokenizer=tokenizer,
    model=protbert_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    window_batch_size=WINDOW_BATCH_SIZE,
    save_every=SAVE_EVERY
)

[train] 25/1274 proteins | embeddings=(25, 1024) | total_windows=32 | avg_windows/protein=1.28 | elapsed=0.01 min
[train] 50/1274 proteins | embeddings=(50, 1024) | total_windows=64 | avg_windows/protein=1.28 | elapsed=0.03 min
[train] 75/1274 proteins | embeddings=(75, 1024) | total_windows=94 | avg_windows/protein=1.25 | elapsed=0.04 min
[train] 100/1274 proteins | embeddings=(100, 1024) | total_windows=127 | avg_windows/protein=1.27 | elapsed=0.06 min
[train] 125/1274 proteins | embeddings=(125, 1024) | total_windows=156 | avg_windows/protein=1.25 | elapsed=0.07 min
[train] 150/1274 proteins | embeddings=(150, 1024) | total_windows=187 | avg_windows/protein=1.25 | elapsed=0.09 min
[train] 175/1274 proteins | embeddings=(175, 1024) | total_windows=222 | avg_windows/protein=1.27 | elapsed=0.10 min
[train] 200/1274 proteins | embeddings=(200, 1024) | total_windows=251 | avg_windows/protein=1.25 | elapsed=0.12 min
[train] 225/1274 proteins | embeddings=(225, 1024) | total_windows=285 | 

In [ ]:
# ============================================================
# LOAD SAVED SLIDING-WINDOW EMBEDDINGS
# ============================================================

X_train_sw_protbert = np.load(EMBED_DIR / "protbert_sw_train_embeddings.npy")
X_val_sw_protbert = np.load(EMBED_DIR / "protbert_sw_val_embeddings.npy")
X_test_sw_protbert = np.load(EMBED_DIR / "protbert_sw_test_embeddings.npy")

meta_train_sw_protbert = pd.read_csv(EMBED_DIR / "protbert_sw_train_metadata.csv")
meta_val_sw_protbert = pd.read_csv(EMBED_DIR / "protbert_sw_val_metadata.csv")
meta_test_sw_protbert = pd.read_csv(EMBED_DIR / "protbert_sw_test_metadata.csv")

y_train = meta_train_sw_protbert["label"].astype(int)
y_val = meta_val_sw_protbert["label"].astype(int)
y_test = meta_test_sw_protbert["label"].astype(int)

print("X_train_sw_protbert:", X_train_sw_protbert.shape)
print("X_val_sw_protbert:", X_val_sw_protbert.shape)
print("X_test_sw_protbert:", X_test_sw_protbert.shape)

print("\nTrain labels:")
print(y_train.value_counts())

display(meta_train_sw_protbert.head())

X_train_sw_protbert: (1274, 1024)
X_val_sw_protbert: (273, 1024)
X_test_sw_protbert: (273, 1024)

Train labels:
label
0    637
1    637
Name: count, dtype: int64


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,n_windows,window_size,stride,representation
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
2,2,ENSG00000143167,GPA33,Q99795,0,319,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022


In [ ]:
# ============================================================
# WINDOW SUMMARY
# ============================================================

protbert_sw_window_summary = []

for split_name, meta in [
    ("train", meta_train_sw_protbert),
    ("validation", meta_val_sw_protbert),
    ("test", meta_test_sw_protbert)
]:
    protbert_sw_window_summary.append({
        "split": split_name,
        "n": len(meta),
        "mean_sequence_length": meta["sequence_clean_length"].mean(),
        "median_sequence_length": meta["sequence_clean_length"].median(),
        "max_sequence_length": meta["sequence_clean_length"].max(),
        "mean_n_windows": meta["n_windows"].mean(),
        "median_n_windows": meta["n_windows"].median(),
        "max_n_windows": meta["n_windows"].max(),
        "n_multi_window_proteins": int((meta["n_windows"] > 1).sum()),
        "pct_multi_window_proteins": (meta["n_windows"] > 1).mean() * 100
    })

protbert_sw_window_summary_df = pd.DataFrame(protbert_sw_window_summary)

display(protbert_sw_window_summary_df)

protbert_sw_window_summary_df.to_csv(
    RESULT_DIR / "protbert_sliding_window_summary_v1.csv",
    index=False
)

,split,n,mean_sequence_length,median_sequence_length,max_sequence_length,mean_n_windows,median_n_windows,max_n_windows,n_multi_window_proteins,pct_multi_window_proteins
0,train,1274,744.512559,555.0,7388,1.267661,1.0,8,263,20.643642
1,validation,273,869.728938,576.0,34350,1.402930,1.0,34,65,23.809524
2,test,273,774.432234,574.0,4861,1.329670,1.0,5,69,25.274725


In [ ]:
# ============================================================
# FREE GPU MEMORY AFTER EMBEDDING
# ============================================================

del protbert_model
torch.cuda.empty_cache()
gc.collect()

print("GPU memory cleared.")

GPU memory cleared.


In [ ]:
# ============================================================
# EVALUATION HELPERS
# ============================================================

def get_positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict(X)


def evaluate_binary_classifier(model, X, y, model_name, dataset_name):
    y_pred = model.predict(X)
    y_score = get_positive_class_score(model, X)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "model": model_name,
        "dataset": dataset_name,

        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, y_pred, zero_division=0),

        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "mcc": matthews_corrcoef(y, y_pred),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [ ]:
# ============================================================
# CV + SCORING
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "mcc": make_scorer(matthews_corrcoef)
}

In [ ]:
# ============================================================
# DOWNSTREAM MODELS FOR PROTBERT SLIDING-WINDOW EMBEDDINGS
# ============================================================

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False


protbert_sw_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])
}


if LIGHTGBM_AVAILABLE:
    protbert_sw_models["LightGBM"] = Pipeline([
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=15,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ])
else:
    protbert_sw_models["Hist Gradient Boosting"] = Pipeline([
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            random_state=42
        ))
    ])

print("ProtBERT sliding-window downstream models:")
for name in protbert_sw_models:
    print("-", name)

ProtBERT sliding-window downstream models:
- Logistic Regression
- SVM RBF
- Random Forest
- LightGBM


In [ ]:
# ============================================================
# BASELINE CV BEFORE TUNING
# ============================================================

protbert_sw_baseline_cv_results = []

for model_name, pipeline in protbert_sw_models.items():
    print("=" * 80)
    print("ProtBERT sliding-window baseline CV:", model_name)

    cv_output = cross_validate(
        estimator=pipeline,
        X=X_train_sw_protbert,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    row = {
        "representation": REPRESENTATION_NAME,
        "model": model_name,

        "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
        "train_roc_auc_std": cv_output["train_roc_auc"].std(),

        "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
        "cv_roc_auc_std": cv_output["test_roc_auc"].std(),

        "cv_f1_mean": cv_output["test_f1"].mean(),
        "cv_f1_std": cv_output["test_f1"].std(),

        "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
        "cv_pr_auc_std": cv_output["test_pr_auc"].std(),

        "cv_mcc_mean": cv_output["test_mcc"].mean(),
        "cv_mcc_std": cv_output["test_mcc"].std(),

        "overfit_gap_train_minus_cv": (
            cv_output["train_roc_auc"].mean() - cv_output["test_roc_auc"].mean()
        )
    }

    protbert_sw_baseline_cv_results.append(row)

protbert_sw_baseline_cv_df = pd.DataFrame(protbert_sw_baseline_cv_results).sort_values(
    by="cv_roc_auc_mean",
    ascending=False
)

display(protbert_sw_baseline_cv_df)

protbert_sw_baseline_cv_df.to_csv(
    RESULT_DIR / "protbert_sw_baseline_cv_before_tuning_v1.csv",
    index=False
)

ProtBERT sliding-window baseline CV: Logistic Regression
ProtBERT sliding-window baseline CV: SVM RBF
ProtBERT sliding-window baseline CV: Random Forest
ProtBERT sliding-window baseline CV: LightGBM


,representation,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap_train_minus_cv
1,ProtBERT_sliding_window_mean_pool_1022_stride_1022,SVM RBF,0.942505,0.002289,0.720996,0.033535,0.656212,0.031949,0.708596,0.032764,0.311530,0.068675,0.221509
2,ProtBERT_sliding_window_mean_pool_1022_stride_1022,Random Forest,1.000000,0.000000,0.710431,0.032637,0.643985,0.024399,0.705531,0.027121,0.287872,0.062173,0.289569
3,ProtBERT_sliding_window_mean_pool_1022_stride_1022,LightGBM,1.000000,0.000000,0.695804,0.037763,0.645157,0.035728,0.688375,0.035878,0.285933,0.067869,0.304196
0,ProtBERT_sliding_window_mean_pool_1022_stride_1022,Logistic Regression,0.999999,0.000002,0.654770,0.035475,0.610233,0.025947,0.649113,0.054968,0.221848,0.040978,0.345229


In [ ]:
# ============================================================
# PARAMETER GRIDS — COLAB OPTIMIZED
# ============================================================

protbert_sw_param_grids = {
    "Logistic Regression": {
        "model__C": [0.0003, 0.001, 0.003, 0.01],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    },

    "SVM RBF": {
        "model__C": [0.1, 1, 10],
        "model__gamma": [0.0001, 0.001, "scale"]
    },

    "Random Forest": {
        "model__n_estimators": [300, 500],
        "model__max_depth": [5, 8, 10],
        "model__min_samples_leaf": [5, 10, 20],
        "model__max_features": ["sqrt", 0.2],
        "model__bootstrap": [True]
    }
}


if LIGHTGBM_AVAILABLE:
    protbert_sw_param_grids["LightGBM"] = {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.03, 0.05],
        "model__num_leaves": [7, 15],
        "model__max_depth": [3, 5],
        "model__min_child_samples": [20, 50],
        "model__subsample": [0.8],
        "model__colsample_bytree": [0.8, 1.0],
        "model__reg_alpha": [0.1],
        "model__reg_lambda": [1, 5]
    }
else:
    protbert_sw_param_grids["Hist Gradient Boosting"] = {
        "model__learning_rate": [0.03, 0.05],
        "model__max_iter": [100, 200],
        "model__max_leaf_nodes": [15, 31],
        "model__min_samples_leaf": [20, 50]
    }

In [25]:
# ============================================================
# RUN GRIDSEARCHCV
# ============================================================

protbert_sw_grid_search_results = []
protbert_sw_best_estimators = {}

for model_name, pipeline in protbert_sw_models.items():
    print("=" * 80)
    print("ProtBERT sliding-window GridSearchCV:", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=protbert_sw_param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train_sw_protbert, y_train)

    protbert_sw_best_estimators[model_name] = grid.best_estimator_

    best_idx = grid.best_index_

    result_row = {
        "representation": REPRESENTATION_NAME,
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_train_roc_auc": grid.cv_results_["mean_train_score"][best_idx],
        "overfit_gap_train_minus_cv": (
            grid.cv_results_["mean_train_score"][best_idx] - grid.best_score_
        ),
        "best_params": grid.best_params_
    }

    protbert_sw_grid_search_results.append(result_row)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best train ROC-AUC:", grid.cv_results_["mean_train_score"][best_idx])
    print("Gap:", result_row["overfit_gap_train_minus_cv"])
    print("Best params:", grid.best_params_)

protbert_sw_grid_results_df = pd.DataFrame(protbert_sw_grid_search_results).sort_values(
    by="best_cv_roc_auc",
    ascending=False
)

display(protbert_sw_grid_results_df)

protbert_sw_grid_results_df.to_csv(
    RESULT_DIR / "protbert_sw_gridsearch_results_v1.csv",
    index=False
)

ProtBERT sliding-window GridSearchCV: Logistic Regression
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV ROC-AUC: 0.7266600533201066
Best train ROC-AUC: 0.8148813113287998
Gap: 0.08822125800869318
Best params: {'model__C': 0.001, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
ProtBERT sliding-window GridSearchCV: SVM RBF
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV ROC-AUC: 0.7209961482422964
Best train ROC-AUC: 0.9425048987505938
Gap: 0.22150875050829744
Best params: {'model__C': 1, 'model__gamma': 'scale'}
ProtBERT sliding-window GridSearchCV: Random Forest
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best CV ROC-AUC: 0.7182266802033604
Best train ROC-AUC: 0.9975864942378969
Gap: 0.27935981403453647
Best params: {'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 500}
ProtBERT sliding-window GridSearchCV: LightGBM
Fitting 5 folds for each o

,representation,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
0,ProtBERT_sliding_window_mean_pool_1022_stride_1022,Logistic Regression,0.726660,0.814881,0.088221,"{'model__C': 0.001, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"
1,ProtBERT_sliding_window_mean_pool_1022_stride_1022,SVM RBF,0.720996,0.942505,0.221509,"{'model__C': 1, 'model__gamma': 'scale'}"
2,ProtBERT_sliding_window_mean_pool_1022_stride_1022,Random Forest,0.718227,0.997586,0.279360,"{'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 500}"
3,ProtBERT_sliding_window_mean_pool_1022_stride_1022,LightGBM,0.710203,0.999714,0.289510,"{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__min_child_samples': 20, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__reg_alpha': 0.1, 'model__reg_lambda': 1, 'model__subsample': 0.8}"


In [26]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

protbert_sw_tuned_eval_records = []

for model_name, model in protbert_sw_best_estimators.items():
    print("=" * 80)
    print("Evaluating:", model_name)

    train_metrics = evaluate_binary_classifier(
        model=model,
        X=X_train_sw_protbert,
        y=y_train,
        model_name=model_name,
        dataset_name="train"
    )

    val_metrics = evaluate_binary_classifier(
        model=model,
        X=X_val_sw_protbert,
        y=y_val,
        model_name=model_name,
        dataset_name="validation"
    )

    train_metrics["representation"] = REPRESENTATION_NAME
    val_metrics["representation"] = REPRESENTATION_NAME

    protbert_sw_tuned_eval_records.extend([train_metrics, val_metrics])

protbert_sw_tuned_eval_df = pd.DataFrame(protbert_sw_tuned_eval_records)

display(protbert_sw_tuned_eval_df)

protbert_sw_validation_comparison = protbert_sw_tuned_eval_df[
    protbert_sw_tuned_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(protbert_sw_validation_comparison)

protbert_sw_tuned_eval_df.to_csv(
    RESULT_DIR / "protbert_sw_tuned_train_validation_metrics_v1.csv",
    index=False
)

protbert_sw_validation_comparison.to_csv(
    RESULT_DIR / "protbert_sw_validation_comparison_v1.csv",
    index=False
)

Evaluating: Logistic Regression
Evaluating: SVM RBF
Evaluating: Random Forest
Evaluating: LightGBM


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,train,0.726845,0.730463,0.718995,0.734694,0.724684,0.810920,0.809877,0.453745,468,169,179,458,ProtBERT_sliding_window_mean_pool_1022_stride_1022
1,Logistic Regression,validation,0.677656,0.696000,0.635036,0.720588,0.664122,0.738622,0.706355,0.356891,98,38,50,87,ProtBERT_sliding_window_mean_pool_1022_stride_1022
2,SVM RBF,train,0.859498,0.866987,0.849294,0.869702,0.858049,0.937376,0.939122,0.719145,554,83,96,541,ProtBERT_sliding_window_mean_pool_1022_stride_1022
3,SVM RBF,validation,0.684982,0.707317,0.635036,0.735294,0.669231,0.731913,0.691923,0.372153,100,36,50,87,ProtBERT_sliding_window_mean_pool_1022_stride_1022
4,Random Forest,train,0.970958,0.973186,0.968603,0.973312,0.970889,0.995626,0.995835,0.941926,620,17,20,617,ProtBERT_sliding_window_mean_pool_1022_stride_1022
5,Random Forest,validation,0.695971,0.717742,0.649635,0.742647,0.681992,0.728424,0.689949,0.393935,101,35,48,89,ProtBERT_sliding_window_mean_pool_1022_stride_1022
6,LightGBM,train,0.987441,0.992076,0.982732,0.992151,0.987382,0.998795,0.998868,0.974926,632,5,11,626,ProtBERT_sliding_window_mean_pool_1022_stride_1022
7,LightGBM,validation,0.644689,0.658730,0.605839,0.683824,0.631179,0.702769,0.677671,0.290522,93,43,54,83,ProtBERT_sliding_window_mean_pool_1022_stride_1022


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
1,Logistic Regression,validation,0.677656,0.696000,0.635036,0.720588,0.664122,0.738622,0.706355,0.356891,98,38,50,87,ProtBERT_sliding_window_mean_pool_1022_stride_1022
3,SVM RBF,validation,0.684982,0.707317,0.635036,0.735294,0.669231,0.731913,0.691923,0.372153,100,36,50,87,ProtBERT_sliding_window_mean_pool_1022_stride_1022
5,Random Forest,validation,0.695971,0.717742,0.649635,0.742647,0.681992,0.728424,0.689949,0.393935,101,35,48,89,ProtBERT_sliding_window_mean_pool_1022_stride_1022
7,LightGBM,validation,0.644689,0.658730,0.605839,0.683824,0.631179,0.702769,0.677671,0.290522,93,43,54,83,ProtBERT_sliding_window_mean_pool_1022_stride_1022


In [27]:
# ============================================================
# SOFT VOTING
# ============================================================

protbert_sw_voting_estimators = []

for model_name, estimator in protbert_sw_best_estimators.items():
    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    protbert_sw_voting_estimators.append((short_name, estimator))

protbert_sw_soft_voting = VotingClassifier(
    estimators=protbert_sw_voting_estimators,
    voting="soft",
    n_jobs=-1
)

protbert_sw_soft_voting.fit(X_train_sw_protbert, y_train)

protbert_sw_voting_train_metrics = evaluate_binary_classifier(
    protbert_sw_soft_voting,
    X_train_sw_protbert,
    y_train,
    "Soft Voting",
    "train"
)

protbert_sw_voting_val_metrics = evaluate_binary_classifier(
    protbert_sw_soft_voting,
    X_val_sw_protbert,
    y_val,
    "Soft Voting",
    "validation"
)

protbert_sw_voting_train_metrics["representation"] = REPRESENTATION_NAME
protbert_sw_voting_val_metrics["representation"] = REPRESENTATION_NAME

display(pd.DataFrame([protbert_sw_voting_train_metrics, protbert_sw_voting_val_metrics]))

protbert_sw_best_estimators["Soft Voting"] = protbert_sw_soft_voting

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Soft Voting,train,0.917582,0.923567,0.910518,0.924647,0.916996,0.976319,0.978271,0.835248,589,48,57,580,ProtBERT_sliding_window_mean_pool_1022_stride_1022
1,Soft Voting,validation,0.695971,0.721311,0.642336,0.750000,0.679537,0.730947,0.688731,0.394566,102,34,49,88,ProtBERT_sliding_window_mean_pool_1022_stride_1022


In [28]:
# ============================================================
# STACKING
# ============================================================

protbert_sw_stacking_estimators = []

for model_name, estimator in protbert_sw_best_estimators.items():
    if model_name == "Soft Voting":
        continue

    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    protbert_sw_stacking_estimators.append((short_name, estimator))

protbert_sw_stacking = StackingClassifier(
    estimators=protbert_sw_stacking_estimators,
    final_estimator=LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

protbert_sw_stacking.fit(X_train_sw_protbert, y_train)

protbert_sw_stacking_train_metrics = evaluate_binary_classifier(
    protbert_sw_stacking,
    X_train_sw_protbert,
    y_train,
    "Stacking",
    "train"
)

protbert_sw_stacking_val_metrics = evaluate_binary_classifier(
    protbert_sw_stacking,
    X_val_sw_protbert,
    y_val,
    "Stacking",
    "validation"
)

protbert_sw_stacking_train_metrics["representation"] = REPRESENTATION_NAME
protbert_sw_stacking_val_metrics["representation"] = REPRESENTATION_NAME

display(pd.DataFrame([protbert_sw_stacking_train_metrics, protbert_sw_stacking_val_metrics]))

protbert_sw_best_estimators["Stacking"] = protbert_sw_stacking

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Stacking,train,0.878336,0.884984,0.869702,0.886970,0.877276,0.951243,0.954700,0.756785,565,72,83,554,ProtBERT_sliding_window_mean_pool_1022_stride_1022
1,Stacking,validation,0.684982,0.707317,0.635036,0.735294,0.669231,0.735562,0.692642,0.372153,100,36,50,87,ProtBERT_sliding_window_mean_pool_1022_stride_1022


In [29]:
# ============================================================
# FINAL VALIDATION COMPARISON
# ============================================================

protbert_sw_all_eval_df = pd.concat(
    [
        protbert_sw_tuned_eval_df,
        pd.DataFrame([
            protbert_sw_voting_train_metrics,
            protbert_sw_voting_val_metrics,
            protbert_sw_stacking_train_metrics,
            protbert_sw_stacking_val_metrics
        ])
    ],
    ignore_index=True
)

protbert_sw_all_validation_results = protbert_sw_all_eval_df[
    protbert_sw_all_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(protbert_sw_all_validation_results)

protbert_sw_all_eval_df.to_csv(
    RESULT_DIR / "protbert_sw_all_train_validation_metrics_v1.csv",
    index=False
)

protbert_sw_all_validation_results.to_csv(
    RESULT_DIR / "protbert_sw_all_validation_comparison_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
1,Logistic Regression,validation,0.677656,0.696000,0.635036,0.720588,0.664122,0.738622,0.706355,0.356891,98,38,50,87,ProtBERT_sliding_window_mean_pool_1022_stride_1022
11,Stacking,validation,0.684982,0.707317,0.635036,0.735294,0.669231,0.735562,0.692642,0.372153,100,36,50,87,ProtBERT_sliding_window_mean_pool_1022_stride_1022
3,SVM RBF,validation,0.684982,0.707317,0.635036,0.735294,0.669231,0.731913,0.691923,0.372153,100,36,50,87,ProtBERT_sliding_window_mean_pool_1022_stride_1022
9,Soft Voting,validation,0.695971,0.721311,0.642336,0.750000,0.679537,0.730947,0.688731,0.394566,102,34,49,88,ProtBERT_sliding_window_mean_pool_1022_stride_1022
5,Random Forest,validation,0.695971,0.717742,0.649635,0.742647,0.681992,0.728424,0.689949,0.393935,101,35,48,89,ProtBERT_sliding_window_mean_pool_1022_stride_1022
7,LightGBM,validation,0.644689,0.658730,0.605839,0.683824,0.631179,0.702769,0.677671,0.290522,93,43,54,83,ProtBERT_sliding_window_mean_pool_1022_stride_1022


In [30]:
# ============================================================
# SELECT FINAL MODEL FROM VALIDATION
# ============================================================

protbert_sw_best_validation_row = protbert_sw_all_validation_results.iloc[0]
final_protbert_sw_model_name = protbert_sw_best_validation_row["model"]
final_protbert_sw_model = protbert_sw_best_estimators[final_protbert_sw_model_name]

print("Final selected ProtBERT sliding-window model:", final_protbert_sw_model_name)
display(protbert_sw_best_validation_row)

pd.DataFrame([protbert_sw_best_validation_row]).to_csv(
    RESULT_DIR / "protbert_sw_final_model_validation_summary_v1.csv",
    index=False
)

Final selected ProtBERT sliding-window model: Logistic Regression


,1
model,Logistic Regression
dataset,validation
accuracy,0.677656
precision,0.696
recall_sensitivity,0.635036
specificity,0.720588
f1,0.664122
roc_auc,0.738622
pr_auc,0.706355
mcc,0.356891


In [31]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

protbert_sw_final_test_metrics = evaluate_binary_classifier(
    model=final_protbert_sw_model,
    X=X_test_sw_protbert,
    y=y_test,
    model_name=final_protbert_sw_model_name,
    dataset_name="test"
)

protbert_sw_final_test_metrics["representation"] = REPRESENTATION_NAME

protbert_sw_final_test_metrics_df = pd.DataFrame([protbert_sw_final_test_metrics])

display(protbert_sw_final_test_metrics_df)

protbert_sw_final_test_metrics_df.to_csv(
    RESULT_DIR / "protbert_sw_final_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,test,0.59707,0.598485,0.580882,0.613139,0.589552,0.648723,0.6551,0.194125,84,53,57,79,ProtBERT_sliding_window_mean_pool_1022_stride_1022


In [32]:
# ============================================================
# DIAGNOSTIC TEST ALL MODELS
# Not used for final model selection.
# ============================================================

protbert_sw_diagnostic_test_records = []

for model_name, model in protbert_sw_best_estimators.items():
    metrics = evaluate_binary_classifier(
        model=model,
        X=X_test_sw_protbert,
        y=y_test,
        model_name=model_name,
        dataset_name="test_diagnostic"
    )

    metrics["representation"] = REPRESENTATION_NAME
    protbert_sw_diagnostic_test_records.append(metrics)

protbert_sw_diagnostic_test_df = pd.DataFrame(protbert_sw_diagnostic_test_records).sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(protbert_sw_diagnostic_test_df)

protbert_sw_diagnostic_test_df.to_csv(
    RESULT_DIR / "protbert_sw_diagnostic_all_models_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
5,Stacking,test_diagnostic,0.589744,0.588235,0.588235,0.591241,0.588235,0.652963,0.659067,0.179476,81,56,56,80,ProtBERT_sliding_window_mean_pool_1022_stride_1022
1,SVM RBF,test_diagnostic,0.582418,0.577465,0.602941,0.562044,0.589928,0.652855,0.654420,0.165118,77,60,54,82,ProtBERT_sliding_window_mean_pool_1022_stride_1022
4,Soft Voting,test_diagnostic,0.600733,0.597122,0.610294,0.591241,0.603636,0.651567,0.657231,0.201567,81,56,53,83,ProtBERT_sliding_window_mean_pool_1022_stride_1022
0,Logistic Regression,test_diagnostic,0.597070,0.598485,0.580882,0.613139,0.589552,0.648723,0.655100,0.194125,84,53,57,79,ProtBERT_sliding_window_mean_pool_1022_stride_1022
3,LightGBM,test_diagnostic,0.608059,0.597315,0.654412,0.562044,0.624561,0.639867,0.623252,0.217367,77,60,47,89,ProtBERT_sliding_window_mean_pool_1022_stride_1022
2,Random Forest,test_diagnostic,0.586081,0.579310,0.617647,0.554745,0.597865,0.634768,0.643473,0.172726,76,61,52,84,ProtBERT_sliding_window_mean_pool_1022_stride_1022


In [33]:
# ============================================================
# SAVE FINAL TEST PREDICTIONS
# ============================================================

y_test_pred = final_protbert_sw_model.predict(X_test_sw_protbert)
y_test_score = get_positive_class_score(final_protbert_sw_model, X_test_sw_protbert)

protbert_sw_test_predictions_df = meta_test_sw_protbert.copy()
protbert_sw_test_predictions_df["true_label"] = y_test.values
protbert_sw_test_predictions_df["pred_label"] = y_test_pred
protbert_sw_test_predictions_df["pred_score_t2d_associated"] = y_test_score

display(protbert_sw_test_predictions_df.head())

protbert_sw_test_predictions_df.to_csv(
    RESULT_DIR / "protbert_sw_final_test_predictions_v1.csv",
    index=False
)

,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,n_windows,window_size,stride,representation,true_label,pred_label,pred_score_t2d_associated
0,0,ENSG00000177542,SLC25A22,Q9H936,0,323,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,0,1,0.758499
1,1,ENSG00000123080,CDKN2C,P42773,1,168,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,1,0,0.464095
2,2,ENSG00000185262,UBALD2,Q8IYN6,0,164,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,0,1,0.806849
3,3,ENSG00000092148,HECTD1,Q9ULT8,0,2610,3,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,0,1,0.613123
4,4,ENSG00000139364,TMEM132B,Q14DG7,1,1078,2,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,1,0,0.493005


In [34]:
# ============================================================
# SAVE MODELS
# ============================================================

for model_name, model in protbert_sw_best_estimators.items():
    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")

    model_path = MODEL_DIR / f"protbert_sw_{safe_name}_best_estimator_v1.pkl"
    joblib.dump(model, model_path)

    print("Saved:", model_path)

final_protbert_sw_model_path = MODEL_DIR / "protbert_sw_final_selected_model_v1.pkl"
joblib.dump(final_protbert_sw_model, final_protbert_sw_model_path)

print("Final ProtBERT sliding-window model saved:", final_protbert_sw_model_path)

Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/models/protbert_sw_logistic_regression_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/models/protbert_sw_svm_rbf_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/models/protbert_sw_random_forest_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/models/protbert_sw_lightgbm_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/models/protbert_sw_soft_voting_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/models/protbert_sw_stacking_best_estimator_v1.pkl
Final ProtBERT sliding-window model saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_slidin

In [35]:
# ============================================================
# MASTER COMPARISON TABLE
# Fill previous numbers from completed phases
# ============================================================

master_rows = [
    {
        "phase": "1.1",
        "representation": "AAC + Physchem",
        "embedding_policy": "handcrafted",
        "final_model": "Random Forest",
        "test_roc_auc": 0.5520,
        "test_pr_auc": 0.5550,
        "test_f1": 0.5390,
        "test_mcc": 0.0480
    },
    {
        "phase": "1.2A",
        "representation": "ESM2_t6_8M",
        "embedding_policy": "truncated_1022",
        "final_model": "Stacking",
        "test_roc_auc": 0.6202,
        "test_pr_auc": 0.6188,
        "test_f1": 0.5926,
        "test_mcc": 0.1941
    },
    {
        "phase": "1.2B",
        "representation": "ESM2_t6_8M",
        "embedding_policy": "sliding_window_1022_stride_1022",
        "final_model": "Soft Voting",
        "test_roc_auc": 0.6277,
        "test_pr_auc": 0.6278,
        "test_f1": 0.5870,
        "test_mcc": 0.1650
    },
    {
        "phase": "1.2C",
        "representation": "ESM2_t12_35M",
        "embedding_policy": "truncated_1022",
        "final_model": "Soft Voting",
        "test_roc_auc": 0.5875,
        "test_pr_auc": 0.5942,
        "test_f1": 0.5654,
        "test_mcc": 0.0995
    },
    {
        "phase": "1.2D",
        "representation": "ProtBERT",
        "embedding_policy": "truncated_1022",
        "final_model": "Logistic Regression",
        "test_roc_auc": 0.6371,
        "test_pr_auc": 0.6423,
        "test_f1": 0.6014,
        "test_mcc": 0.1943
    },
    {
        "phase": "1.2E",
        "representation": "ProtBERT",
        "embedding_policy": "sliding_window_1022_stride_1022",
        "final_model": final_protbert_sw_model_name,
        "test_roc_auc": protbert_sw_final_test_metrics["roc_auc"],
        "test_pr_auc": protbert_sw_final_test_metrics["pr_auc"],
        "test_f1": protbert_sw_final_test_metrics["f1"],
        "test_mcc": protbert_sw_final_test_metrics["mcc"]
    }
]

master_comparison_df = pd.DataFrame(master_rows)

display(master_comparison_df)

master_comparison_df.to_csv(
    RESULT_DIR / "master_comparison_with_protbert_sliding_window_v1.csv",
    index=False
)

,phase,representation,embedding_policy,final_model,test_roc_auc,test_pr_auc,test_f1,test_mcc
0,1.1,AAC + Physchem,handcrafted,Random Forest,0.552000,0.5550,0.539000,0.048000
1,1.2A,ESM2_t6_8M,truncated_1022,Stacking,0.620200,0.6188,0.592600,0.194100
2,1.2B,ESM2_t6_8M,sliding_window_1022_stride_1022,Soft Voting,0.627700,0.6278,0.587000,0.165000
3,1.2C,ESM2_t12_35M,truncated_1022,Soft Voting,0.587500,0.5942,0.565400,0.099500
4,1.2D,ProtBERT,truncated_1022,Logistic Regression,0.637100,0.6423,0.601400,0.194300
5,1.2E,ProtBERT,sliding_window_1022_stride_1022,Logistic Regression,0.648723,0.6551,0.589552,0.194125
